# **Housing Dataset** con ML y  Red Neuronal

- Nicolas Rodríguez Forero
- Daniel Velasco González

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [2]:
df = pd.read_csv('https://raw.githubusercontent.com/camilousa/datasets/refs/heads/master/housing.csv')

In [3]:
df.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   longitude           20640 non-null  float64
 1   latitude            20640 non-null  float64
 2   housing_median_age  20640 non-null  float64
 3   total_rooms         20640 non-null  float64
 4   total_bedrooms      20433 non-null  float64
 5   population          20640 non-null  float64
 6   households          20640 non-null  float64
 7   median_income       20640 non-null  float64
 8   median_house_value  20640 non-null  float64
 9   ocean_proximity     20640 non-null  str    
dtypes: float64(9), str(1)
memory usage: 1.6 MB


### **Preprocesamiento**

In [5]:
# Fill missing values with median
df['total_bedrooms'] = df['total_bedrooms'].fillna(df['total_bedrooms'].median())

# IQR-based outlier removal on all numerical columns
numerical_cols = ['longitude', 'latitude', 'housing_median_age', 'total_rooms',
                  'total_bedrooms', 'population', 'households', 'median_income', 'median_house_value']

for col in numerical_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    df = df[(df[col] >= lower) & (df[col] <= upper)]

print(f"Registros después del filtrado IQR: {len(df)}")
df.head()

Registros después del filtrado IQR: 16896


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY
5,-122.25,37.85,52.0,919.0,213.0,413.0,193.0,4.0368,269700.0,NEAR BAY
6,-122.25,37.84,52.0,2535.0,489.0,1094.0,514.0,3.6591,299200.0,NEAR BAY


In [6]:
feature_cols = ['longitude', 'latitude', 'housing_median_age', 'total_rooms',
                'total_bedrooms', 'population', 'households', 'median_income']

scaler = StandardScaler()
X = pd.DataFrame(scaler.fit_transform(df[feature_cols]), columns=feature_cols)
y = df['median_house_value'].values

print(f"Shape de X: {X.shape}")
print(f"Shape de y: {y.shape}")
X.head()

Shape de X: (16896, 8)
Shape de y: (16896,)


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income
0,-1.309401,0.982511,1.829115,-0.589961,-1.187381,-1.188993,-1.194809,2.545697
1,-1.314387,0.982511,1.829115,-0.778200,-0.961785,-1.076214,-0.969211,1.429274
2,-1.314387,0.982511,1.829115,-0.433908,-0.736188,-1.063481,-0.754355,0.186567
3,-1.314387,0.982511,1.829115,-1.124442,-1.072076,-1.339972,-1.108867,0.318383
4,-1.314387,0.977910,1.829115,0.451692,0.311582,-0.101223,0.615350,0.057172


In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

### Modelo **DecisionTree Regressor**

In [8]:
dt_regressor = DecisionTreeRegressor(
    random_state=42,
    max_depth=5
)

dt_regressor.fit(X_train, y_train)

,"criterion criterion: {""squared_error"", ""friedman_mse"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in the half mean Poisson deviance to find splits... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 0.24 Poisson deviance criterion.",'squared_error'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.For an example of how ``max_depth`` influences the model, see:ref:`sphx_glr_auto_examples_tree_plot_tree_regression.py`.",5
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary ` for details.",42
,"max_leaf

In [9]:
# Make predictions on the training set
y_train_pred_dt = dt_regressor.predict(X_train)

# Make predictions on the test set
y_test_pred_dt = dt_regressor.predict(X_test)

In [10]:
# Calculate metrics for training set
mae_train_dt = mean_absolute_error(y_train, y_train_pred_dt)
mse_train_dt = mean_squared_error(y_train, y_train_pred_dt)
rmse_train_dt = np.sqrt(mse_train_dt)
r2_train_dt = r2_score(y_train, y_train_pred_dt)

# Calculate metrics for test set
mae_test_dt = mean_absolute_error(y_test, y_test_pred_dt)
mse_test_dt = mean_squared_error(y_test, y_test_pred_dt)
rmse_test_dt = np.sqrt(mse_test_dt)
r2_test_dt = r2_score(y_test, y_test_pred_dt)

# Create a DataFrame for Decision Tree results
dt_results = {
    'Métrica': ['MAE', 'MSE', 'RMSE', 'R²'],
    'Entrenamiento': [mae_train_dt, mse_train_dt, rmse_train_dt, r2_train_dt],
    'Prueba': [mae_test_dt, mse_test_dt, rmse_test_dt, r2_test_dt]
}
dt_results_df = pd.DataFrame(dt_results)

print("\nDecision Tree Regressor Performance:")
dt_results_df.head()


Decision Tree Regressor Performance:


,Métrica,Entrenamiento,Prueba
0,MAE,4.684660e+04,4.732165e+04
1,MSE,3.966244e+09,4.120161e+09
2,RMSE,6.297812e+04,6.418848e+04
3,R²,5.354566e-01,5.048299e-01


### **Red Neuronal**

In [11]:
neural_network = Sequential()
neural_network.add(Dense(8, activation='relu', input_shape=(8,)))
neural_network.add(Dense(16, activation='relu'))
neural_network.add(Dense(1, activation='linear'))

neural_network.compile(optimizer='adam', loss='mse', metrics=['mae'])

/Users/daniv/dev/USA/neural-networks/.venv/lib/python3.13/site-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [12]:
neural_network.fit(X_train, y_train, epochs=100, batch_size=32, verbose=0)

In [13]:
y_train_pred_nn = neural_network.predict(X_train, verbose=0).flatten()
y_test_pred_nn = neural_network.predict(X_test, verbose=0).flatten()

In [14]:
def get_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return [
        mean_absolute_error(y_true, y_pred),
        mse,
        np.sqrt(mse),
        r2_score(y_true, y_pred)
    ]

nn_results = {
    'Métrica': ['MAE', 'MSE', 'RMSE', 'R²'],
    'Entrenamiento': get_metrics(y_train, y_train_pred_nn),
    'Prueba': get_metrics(y_test, y_test_pred_nn)
}

nn_results_df = pd.DataFrame(nn_results)

print("\nRendimiento de la Red Neuronal (Linear - 1 Neurona):")
nn_results_df.head()


Rendimiento de la Red Neuronal (Linear - 1 Neurona):


,Métrica,Entrenamiento,Prueba
0,MAE,4.164750e+04,4.191802e+04
1,MSE,3.091514e+09,3.217349e+09
2,RMSE,5.560139e+04,5.672168e+04
3,R²,6.379087e-01,6.133319e-01


### **Resultados Finales**

In [15]:
results_df_ren = dt_results_df.rename(columns={
    'Entrenamiento': 'Entrenamiento_DT',
    'Prueba': 'Prueba_DT'
})

nn_results_df_ren = nn_results_df.rename(columns={
    'Entrenamiento': 'Entrenamiento_NN',
    'Prueba': 'Prueba_NN'
})

final_df = results_df_ren.merge(nn_results_df_ren, on='Métrica')

print("\nResultados comparativos Árbol de Decisión vs Red Neuronal:")
display(final_df)


Resultados comparativos Árbol de Decisión vs Red Neuronal:


,Métrica,Entrenamiento_DT,Prueba_DT,Entrenamiento_NN,Prueba_NN
0,MAE,4.684660e+04,4.732165e+04,4.164750e+04,4.191802e+04
1,MSE,3.966244e+09,4.120161e+09,3.091514e+09,3.217349e+09
2,RMSE,6.297812e+04,6.418848e+04,5.560139e+04,5.672168e+04
3,R²,5.354566e-01,5.048299e-01,6.379087e-01,6.133319e-01
